# 📬 Job Application Email Tracker

Reads inbox (Gmail or Outlook), finds job-application emails, and builds a tidy pandas DataFrame with columns:

| Job Title | Company | Date Applied | Date Last Updated | Status |
|-----------|---------|--------------|-------------------|--------|

**Sections**
1. [Setup - installs & imports](#1-setup)
2. [Configuration](#2-configuration)
3. [Function definitions](#3-function-definitions)
4. [Run the tracker](#4-run)

---
## 1. Setup
Install dependencies and import everything needed.

In [1]:
# ── Install dependencies ────────────────────────────────────────────────────
# Gmail
%pip install -q google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client

# Outlook (Microsoft Graph)
%pip install -q msal requests

# Core
%pip install -q pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: C:\Users\RAZER\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: C:\Users\RAZER\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.1 -> 26.0.1
[notice] To update, run: C:\Users\RAZER\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ── Imports ─────────────────────────────────────────────────────────────────
from __future__ import annotations

import re
import base64
import logging
from abc import ABC, abstractmethod
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from typing import Any, Optional

import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

print("✅ Imports ready.")

✅ Imports ready.


---
## 2. Configuration
Edit the values in this cell before running the tracker.

In [3]:
# ── Choose your provider ─────────────────────────────────────────────────────
# Options: "gmail" | "outlook"
EMAIL_PROVIDER = "gmail"

# ── Gmail settings (only needed when EMAIL_PROVIDER = "gmail") ───────────────
GMAIL_CREDENTIALS_PATH = "credentials.json"   # downloaded from Google Cloud Console
GMAIL_TOKEN_PATH       = "gmail_token.json"   # cached after first auth; auto-created

# ── Outlook settings (only needed when EMAIL_PROVIDER = "outlook") ───────────
OUTLOOK_CLIENT_ID  = "YOUR_AZURE_APP_CLIENT_ID"
OUTLOOK_TENANT_ID  = "common"   # use 'common' for personal Microsoft accounts

# ── Fetch settings ────────────────────────────────────────────────────────────
MAX_EMAILS   = 300                                  # hard cap on emails to fetch
AFTER_DATE   = datetime(2024, 9, 1, tzinfo=timezone.utc)  # ignore emails before this date
LOCAL_TZ     = "America/Toronto"                    # timezone for date display

# ── Classification settings ───────────────────────────────────────────────────
MIN_CONFIDENCE = 0.3   # 0–1; lower = more emails, more noise

# ── Output settings ───────────────────────────────────────────────────────────
INCLUDE_METADATA = True    # adds Email ID / Subject / Confidence debug columns
DEDUPLICATE      = True    # merge same-company+title threads into one row
SAVE_CSV         = True    # write results to CSV
CSV_PATH         = "job_applications.csv"

print("✅ Configuration loaded.")

✅ Configuration loaded.


---
## 3. Function Definitions

All logic lives here, organised into the following layers:

```
EmailProvider  (abstract base)
  ├── GmailProvider
  └── OutlookProvider

RawEmail            ← provider-agnostic email envelope
JobApplication      ← one output row

EmailFetcher        ← wraps a provider; applies default search query
JobEmailClassifier  ← scores emails 0–1 on job-relevance
JobEmailParser      ← extracts title / company / status
deduplicate_applications()
build_dataframe()
run_tracker()       ← orchestrates everything
```

In [4]:
# ── 3.1  Data models ─────────────────────────────────────────────────────────

@dataclass
class RawEmail:
    """Provider-agnostic envelope for a single email message."""
    message_id:  str
    subject:     str
    sender:      str
    recipient:   str
    date_sent:   datetime
    body_text:   str          # plain-text body (HTML stripped by provider)
    body_html:   str = ""
    thread_id:   str = ""
    labels:      list[str] = field(default_factory=list)
    provider:    str = "unknown"


@dataclass
class JobApplication:
    """One row in the output DataFrame."""
    job_title:          str = ""
    company:            str = ""
    date_applied:       Optional[datetime] = None
    date_last_updated:  Optional[datetime] = None
    status:             str = "Unknown"
    source_email_id:    str = ""
    source_subject:     str = ""
    confidence:         float = 0.0

print("✅ Data models defined.")

✅ Data models defined.


In [5]:
# ── 3.2  Email provider - abstract base ──────────────────────────────────────

class EmailProvider(ABC):
    """
    Abstract base class for all inbox sources.
    Implement authenticate() and fetch_emails() to add a new provider.
    """

    @abstractmethod
    def authenticate(self) -> None:
        """Run the OAuth / credential flow for this provider."""

    @abstractmethod
    def fetch_emails(
        self,
        query:       str = "",
        max_results: int = 100,
        after_date:  Optional[datetime] = None,
    ) -> list[RawEmail]:
        """
        Fetch emails matching *query*.

        Parameters
        ----------
        query       : Provider-native search string.
        max_results : Hard cap on number of messages to return.
        after_date  : Ignore emails older than this timestamp.
        """

print("✅ EmailProvider (ABC) defined.")

✅ EmailProvider (ABC) defined.


In [6]:
# ── 3.3  Gmail provider ───────────────────────────────────────────────────────
#
# Prerequisites
# -------------
# 1. Create a project at https://console.cloud.google.com
# 2. Enable the Gmail API
# 3. Create OAuth 2.0 credentials (type: Desktop App)
# 4. Download the JSON as `credentials.json` (or update GMAIL_CREDENTIALS_PATH above)
#
# The first call to authenticate() opens a browser window.
# The token is cached at GMAIL_TOKEN_PATH; subsequent runs skip the browser.

class GmailProvider(EmailProvider):
    """
    Fetches emails via the Gmail REST API using OAuth2.

    Parameters
    ----------
    credentials_path : Path to the OAuth2 client_secret JSON from Google Cloud.
    token_path       : Where to cache the user token between runs.
    scopes           : Gmail API scopes (defaults to read-only).
    """

    DEFAULT_SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]

    def __init__(
        self,
        credentials_path: str = "credentials.json",
        token_path:       str = "gmail_token.json",
        scopes:           Optional[list[str]] = None,
    ) -> None:
        self.credentials_path = credentials_path
        self.token_path       = token_path
        self.scopes           = scopes or self.DEFAULT_SCOPES
        self._service: Any    = None

    # ------------------------------------------------------------------
    def authenticate(self) -> None:
        """Run OAuth2 flow; caches token to *token_path* after first run."""
        import os
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials
        from google_auth_oauthlib.flow import InstalledAppFlow
        from googleapiclient.discovery import build

        creds = None
        if os.path.exists(self.token_path):
            creds = Credentials.from_authorized_user_file(self.token_path, self.scopes)

        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(
                    self.credentials_path, self.scopes
                )
                creds = flow.run_local_server(port=0)
            with open(self.token_path, "w") as fh:
                fh.write(creds.to_json())

        self._service = build("gmail", "v1", credentials=creds)
        logger.info("Gmail authenticated successfully.")

    # ------------------------------------------------------------------
    def fetch_emails(
        self,
        query:       str = "",
        max_results: int = 100,
        after_date:  Optional[datetime] = None,
    ) -> list[RawEmail]:
        """Fetch Gmail messages matching *query*."""
        if self._service is None:
            raise RuntimeError("Call authenticate() before fetch_emails().")

        if after_date:
            query = f"{query} after:{after_date.strftime('%Y/%m/%d')}".strip()

        response = (
            self._service.users()
            .messages()
            .list(userId="me", q=query, maxResults=max_results)
            .execute()
        )
        stubs = response.get("messages", [])
        logger.info("Gmail: %d message stubs retrieved.", len(stubs))

        emails: list[RawEmail] = []
        for stub in stubs:
            try:
                msg = (
                    self._service.users()
                    .messages()
                    .get(userId="me", id=stub["id"], format="full")
                    .execute()
                )
                emails.append(self._parse_message(msg))
            except Exception as exc:
                logger.warning("Skipping message %s: %s", stub["id"], exc)

        return emails

    # ------------------------------------------------------------------
    @staticmethod
    def _parse_message(msg: dict) -> RawEmail:
        """Convert a Gmail API message dict → RawEmail."""
        headers = {
            h["name"].lower(): h["value"]
            for h in msg.get("payload", {}).get("headers", [])
        }
        try:
            date_sent = parsedate_to_datetime(headers.get("date", ""))
            if date_sent.tzinfo is None:
                date_sent = date_sent.replace(tzinfo=timezone.utc)
        except Exception:
            date_sent = datetime.now(timezone.utc)

        body_text, body_html = GmailProvider._extract_body(msg.get("payload", {}))

        return RawEmail(
            message_id=msg["id"],
            subject=headers.get("subject", "(no subject)"),
            sender=headers.get("from", ""),
            recipient=headers.get("to", ""),
            date_sent=date_sent,
            body_text=body_text,
            body_html=body_html,
            thread_id=msg.get("threadId", ""),
            labels=msg.get("labelIds", []),
            provider="gmail",
        )

    @staticmethod
    def _extract_body(payload: dict) -> tuple[str, str]:
        """Recursively extract plain-text and HTML body parts."""
        text, html = "", ""
        mime = payload.get("mimeType", "")

        if mime == "text/plain":
            data = payload.get("body", {}).get("data", "")
            text = base64.urlsafe_b64decode(data + "==").decode("utf-8", errors="replace")
        elif mime == "text/html":
            data = payload.get("body", {}).get("data", "")
            html = base64.urlsafe_b64decode(data + "==").decode("utf-8", errors="replace")
        else:
            for part in payload.get("parts", []):
                t, h = GmailProvider._extract_body(part)
                text += t
                html += h

        return text, html

print("✅ GmailProvider defined.")

✅ GmailProvider defined.


In [7]:
# ── 3.4  Outlook provider ─────────────────────────────────────────────────────
#
# Prerequisites
# -------------
# 1. Register an app at https://portal.azure.com → App Registrations
# 2. Add "Mail.Read" under API permissions (Microsoft Graph → Delegated)
# 3. Copy the Application (client) ID into OUTLOOK_CLIENT_ID above
# 4. Use OUTLOOK_TENANT_ID = "common" for personal @outlook.com / @hotmail.com accounts
#
# authenticate() uses a device-code flow: it prints a URL and a one-time code;
# open the URL in any browser, enter the code, and sign in.

class OutlookProvider(EmailProvider):
    """
    Fetches emails via Microsoft Graph API using MSAL device-code OAuth2.

    Parameters
    ----------
    client_id        : Azure app (client) ID.
    tenant_id        : Azure tenant ID; use 'common' for personal accounts.
    token_cache_path : Where to persist the MSAL token cache between runs.
    """

    _SCOPES    = ["Mail.Read"]
    _GRAPH_URL = "https://graph.microsoft.com/v1.0"

    def __init__(
        self,
        client_id:        str,
        tenant_id:        str = "common",
        token_cache_path: str = "outlook_token.json",
    ) -> None:
        self.client_id        = client_id
        self.tenant_id        = tenant_id
        self.token_cache_path = token_cache_path
        self._access_token: Optional[str] = None

    # ------------------------------------------------------------------
    def authenticate(self) -> None:
        """Device-code flow - prints a URL+code for the user to enter in a browser."""
        import os, msal

        cache = msal.SerializableTokenCache()
        if os.path.exists(self.token_cache_path):
            cache.deserialize(open(self.token_cache_path).read())

        app = msal.PublicClientApplication(
            self.client_id,
            authority=f"https://login.microsoftonline.com/{self.tenant_id}",
            token_cache=cache,
        )

        accounts = app.get_accounts()
        result = app.acquire_token_silent(self._SCOPES, account=accounts[0]) if accounts else None

        if not result:
            flow  = app.initiate_device_flow(scopes=self._SCOPES)
            print(flow["message"])          # user visits URL and enters code
            result = app.acquire_token_by_device_flow(flow)

        if "access_token" not in result:
            raise RuntimeError(f"Outlook auth failed: {result.get('error_description')}")

        self._access_token = result["access_token"]
        if cache.has_state_changed:
            open(self.token_cache_path, "w").write(cache.serialize())
        logger.info("Outlook authenticated successfully.")

    # ------------------------------------------------------------------
    def fetch_emails(
        self,
        query:       str = "",
        max_results: int = 100,
        after_date:  Optional[datetime] = None,
    ) -> list[RawEmail]:
        """Fetch Outlook/Exchange messages via MS Graph."""
        if not self._access_token:
            raise RuntimeError("Call authenticate() before fetch_emails().")

        import requests

        headers = {"Authorization": f"Bearer {self._access_token}"}
        params: dict[str, Any] = {
            "$top":    max_results,
            "$select": "id,subject,from,toRecipients,receivedDateTime,body",
        }

        filters = []
        if after_date:
            filters.append(f"receivedDateTime ge {after_date.strftime('%Y-%m-%dT%H:%M:%SZ')}")
        if query:
            params["$search"] = f'"{query}"'
        if filters:
            params["$filter"] = " and ".join(filters)

        resp = requests.get(
            f"{self._GRAPH_URL}/me/messages",
            headers=headers,
            params=params,
            timeout=30,
        )
        resp.raise_for_status()
        items = resp.json().get("value", [])
        logger.info("Outlook: %d messages retrieved.", len(items))
        return [self._parse_message(m) for m in items]

    # ------------------------------------------------------------------
    @staticmethod
    def _parse_message(msg: dict) -> RawEmail:
        """Convert an MS Graph message dict → RawEmail."""
        sender_obj = msg.get("from", {}).get("emailAddress", {})
        sender     = f"{sender_obj.get('name', '')} <{sender_obj.get('address', '')}>"
        recipients = ", ".join(
            r["emailAddress"]["address"]
            for r in msg.get("toRecipients", [])
        )
        body_content = msg.get("body", {}).get("content", "")
        body_type    = msg.get("body", {}).get("contentType", "text")

        try:
            date_sent = datetime.fromisoformat(
                msg["receivedDateTime"].replace("Z", "+00:00")
            )
        except Exception:
            date_sent = datetime.now(timezone.utc)

        return RawEmail(
            message_id=msg["id"],
            subject=msg.get("subject", "(no subject)"),
            sender=sender,
            recipient=recipients,
            date_sent=date_sent,
            body_text=body_content if body_type == "text" else "",
            body_html=body_content if body_type == "html" else "",
            provider="outlook",
        )

print("✅ OutlookProvider defined.")

✅ OutlookProvider defined.


In [8]:
# ── 3.5  EmailFetcher ─────────────────────────────────────────────────────────

class EmailFetcher:
    """
    Thin wrapper around any EmailProvider.
    Applies a sensible default job-hunting search query if none is supplied.

    Parameters
    ----------
    provider      : An authenticated EmailProvider instance.
    default_query : Fallback search string used when query=None at call time.
    max_results   : Default page size passed to the provider.
    """

    DEFAULT_JOB_QUERY = (
        "subject:(application OR applied OR interview OR offer OR "
        "rejection OR opportunity OR position OR role OR hiring OR "
        '"thank you for applying" OR "we received your application")'
    )

    def __init__(
        self,
        provider:      EmailProvider,
        default_query: str = DEFAULT_JOB_QUERY,
        max_results:   int = 200,
    ) -> None:
        self.provider      = provider
        self.default_query = default_query
        self.max_results   = max_results

    def fetch(
        self,
        query:       Optional[str]      = None,
        max_results: Optional[int]      = None,
        after_date:  Optional[datetime] = None,
    ) -> list[RawEmail]:
        """
        Fetch emails; falls back to constructor defaults when args are None.

        Parameters
        ----------
        query       : Override the default job-search query.
        max_results : Override the default page size.
        after_date  : Only return emails after this date.
        """
        return self.provider.fetch_emails(
            query=query if query is not None else self.default_query,
            max_results=max_results if max_results is not None else self.max_results,
            after_date=after_date,
        )

print("✅ EmailFetcher defined.")

✅ EmailFetcher defined.


In [9]:
# ── 3.6  JobEmailClassifier ───────────────────────────────────────────────────

class JobEmailClassifier:
    """
    Scores emails on how likely they are job-application related.
    Returns a confidence float in [0, 1]; drops emails below min_confidence.

    Parameters
    ----------
    min_confidence   : Emails scoring below this threshold are discarded.
    subject_keywords : Additional regex patterns to boost the subject score.
    body_keywords    : Additional regex patterns to boost the body score.
    """

    _SUBJECT_SIGNALS = [
        r"\bapplication\b", r"\bapplied\b", r"\binterview\b",
        r"\boffer\b",       r"\brejection\b", r"\bopportunity\b",
        r"\bposition\b",    r"\bhiring\b",    r"\brecruiter\b",
        r"thank you for applying", r"we received your application",
        r"next steps", r"assessment", r"take-home",
        r"candidate",  r"screening",  r"shortlisted",
    ]

    _BODY_SIGNALS = [
        r"we received your application", r"thank you for (your )?interest",
        r"we regret to inform",          r"we('d| would) like to (invite|schedule)",
        r"congratulations.*offer",       r"background check",
        r"start date",                   r"compensation package",
        r"unfortunately.*position",      r"moved forward with other candidates",
    ]

    def __init__(
        self,
        min_confidence:   float = 0.3,
        subject_keywords: Optional[list[str]] = None,
        body_keywords:    Optional[list[str]] = None,
    ) -> None:
        self.min_confidence = min_confidence
        subject_patterns    = self._SUBJECT_SIGNALS + (subject_keywords or [])
        body_patterns       = self._BODY_SIGNALS    + (body_keywords    or [])
        self._subject_re    = [re.compile(p, re.I) for p in subject_patterns]
        self._body_re       = [re.compile(p, re.I) for p in body_patterns]

    def score(self, email: RawEmail) -> float:
        """Return a confidence score in [0, 1] for a single email."""
        subject_hits = sum(1 for r in self._subject_re if r.search(email.subject))
        body_hits    = sum(1 for r in self._body_re    if r.search(email.body_text))
        raw = min(1.0, subject_hits * 0.06 + body_hits * 0.04
                  + (0.30 if subject_hits > 0 else 0))
        return round(raw, 3)

    def filter(
        self,
        emails: list[RawEmail],
    ) -> list[tuple[RawEmail, float]]:
        """
        Return (email, confidence) tuples for emails that exceed *min_confidence*,
        sorted highest-confidence first.
        """
        results = [
            (email, self.score(email))
            for email in emails
            if self.score(email) >= self.min_confidence
        ]
        results.sort(key=lambda x: x[1], reverse=True)
        logger.info("%d / %d emails passed classification.", len(results), len(emails))
        return results

print("✅ JobEmailClassifier defined.")

✅ JobEmailClassifier defined.


In [10]:
# ── 3.7  JobEmailParser ───────────────────────────────────────────────────────

class JobEmailParser:
    """
    Extracts structured fields from a classified email using regex heuristics.

    Parameters
    ----------
    status_map              : Dict mapping regex patterns → canonical status labels.
                              Pass your own to override defaults.
    unknown_company_fallback: Placeholder when company cannot be determined.
    """

    DEFAULT_STATUS_MAP: dict[str, str] = {
        r"offer":                                           "Offer Received",
        r"accept":                                          "Offer Accepted",
        r"congratulations":                                 "Offer Received",
        r"start date":                                      "Offer Accepted",
        r"reject|unfortunately|regret|not moving forward|other candidates": "Rejected",
        r"interview":                                       "Interview Scheduled",
        r"technical|assessment|take.home|coding challenge": "Assessment",
        r"screening|phone screen":                          "Phone Screen",
        r"shortlist|shortlisted":                           "Shortlisted",
        r"application received|we received your":           "Applied",
        r"withdrawn|withdraw":                              "Withdrawn",
    }

    _TITLE_PATTERNS = [
        r"(?:position|role|title|job)[:\s]+([A-Z][^\n,;]{3,60})",
        r"(?:for the|for a|re:|regarding)[:\s]+([A-Z][^\n,;]{3,60}?)(?:\s+at|\s+with|\s+role|\s+-|\s*$)",
        r"([A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,4})\s+(?:position|role|engineer|developer|manager|analyst|designer|scientist)",
    ]

    def __init__(
        self,
        status_map:               Optional[dict[str, str]] = None,
        unknown_company_fallback: str = "Unknown Company",
    ) -> None:
        raw_map = status_map or self.DEFAULT_STATUS_MAP
        self._status_rules = [
            (re.compile(pattern, re.I), label)
            for pattern, label in raw_map.items()
        ]
        self._title_re               = [re.compile(p) for p in self._TITLE_PATTERNS]
        self.unknown_company_fallback = unknown_company_fallback

    # ------------------------------------------------------------------
    def parse(self, email: RawEmail, confidence: float = 0.0) -> JobApplication:
        """Parse a single RawEmail into a JobApplication."""
        return JobApplication(
            job_title          = self._extract_title(email.subject, email.body_text),
            company            = self._extract_company(email.sender, f"{email.subject}\n{email.body_text}"),
            date_applied       = email.date_sent,
            date_last_updated  = email.date_sent,
            status             = self._extract_status(f"{email.subject}\n{email.body_text}"),
            source_email_id    = email.message_id,
            source_subject     = email.subject,
            confidence         = confidence,
        )

    # ------------------------------------------------------------------
    def _extract_title(self, subject: str, body: str) -> str:
        for pattern in self._title_re:
            m = pattern.search(subject) or pattern.search(body)
            if m:
                return m.group(1).strip()
        cleaned = re.sub(r"^(re|fwd|fw)[:\s]+", "", subject, flags=re.I).strip()
        return cleaned[:80] if cleaned else "Unknown Title"

    def _extract_company(self, sender: str, text: str) -> str:
        m = re.search(
            r"\bat\s+([A-Z][A-Za-z0-9&., ]{2,40}?)(?:\s*[,.]|\s+for|\s+we|\s*$)", text
        )
        if m:
            return m.group(1).strip()
        domain_m = re.search(r"@([\w.-]+)", sender)
        if domain_m:
            domain = domain_m.group(1)
            if not any(d in domain for d in ("gmail", "yahoo", "outlook", "hotmail", "icloud")):
                return domain.split(".")[0].capitalize()
        return self.unknown_company_fallback

    def _extract_status(self, text: str) -> str:
        for pattern, label in self._status_rules:
            if pattern.search(text):
                return label
        return "Applied"

print("✅ JobEmailParser defined.")

✅ JobEmailParser defined.


In [11]:
# ── 3.8  Deduplication ────────────────────────────────────────────────────────

def deduplicate_applications(
    apps:                 list[JobApplication],
    group_by:             tuple[str, ...] = ("company", "job_title"),
    prefer_latest_status: bool = True,
) -> list[JobApplication]:
    """
    Merge duplicate applications (same company + title) into one row.
    Keeps the *earliest* date_applied and *latest* date_last_updated.

    Parameters
    ----------
    apps                 : Raw list, possibly containing duplicates from email threads.
    group_by             : Fields used as the dedup key.
    prefer_latest_status : When True, the status of the most-recent email wins.
    """
    buckets: dict[tuple, list[JobApplication]] = defaultdict(list)
    for app in apps:
        key = tuple(getattr(app, f, "").lower().strip() for f in group_by)
        buckets[key].append(app)

    merged: list[JobApplication] = []
    for group in buckets.values():
        group.sort(key=lambda a: a.date_applied or datetime.min.replace(tzinfo=timezone.utc))
        base = group[0]
        for later in group[1:]:
            if later.date_applied and (
                not base.date_applied or later.date_applied < base.date_applied
            ):
                base.date_applied = later.date_applied
            if later.date_last_updated and (
                not base.date_last_updated or later.date_last_updated > base.date_last_updated
            ):
                base.date_last_updated = later.date_last_updated
                if prefer_latest_status:
                    base.status = later.status
        merged.append(base)

    logger.info("Deduplication: %d → %d rows.", len(apps), len(merged))
    return merged

print("✅ deduplicate_applications defined.")

✅ deduplicate_applications defined.


In [12]:
# ── 3.9  DataFrame builder ────────────────────────────────────────────────────

def build_dataframe(
    applications:     list[JobApplication],
    include_metadata: bool = False,
    tz_localize:      Optional[str] = "UTC",
) -> pd.DataFrame:
    """
    Convert a list of JobApplication objects to a tidy pandas DataFrame.

    Parameters
    ----------
    applications     : Parsed (and optionally deduped) JobApplication list.
    include_metadata : Add debug columns: Email ID, Email Subject, Confidence.
    tz_localize      : Timezone label for datetime display (None = leave as UTC).
    """
    records = []
    for app in applications:
        row: dict[str, Any] = {
            "Job Title":         app.job_title,
            "Company":           app.company,
            "Date Applied":      app.date_applied,
            "Date Last Updated": app.date_last_updated,
            "Status":            app.status,
        }
        if include_metadata:
            row["Email ID"]      = app.source_email_id
            row["Email Subject"] = app.source_subject
            row["Confidence"]    = app.confidence
        records.append(row)

    df = pd.DataFrame(records)

    for col in ("Date Applied", "Date Last Updated"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
            if tz_localize and tz_localize != "UTC":
                df[col] = df[col].dt.tz_convert(tz_localize)

    base_cols  = ["Job Title", "Company", "Date Applied", "Date Last Updated", "Status"]
    extra_cols = [c for c in df.columns if c not in base_cols]
    return df[base_cols + extra_cols]

print("✅ build_dataframe defined.")

✅ build_dataframe defined.


In [13]:
# ── 3.10  run_tracker - main orchestrator ─────────────────────────────────────

def run_tracker(
    provider:          EmailProvider,
    fetch_query:       Optional[str]      = None,
    max_emails:        int                = 200,
    after_date:        Optional[datetime] = None,
    min_confidence:    float              = 0.3,
    deduplicate:       bool               = True,
    include_metadata:  bool               = False,
    local_timezone:    str                = "America/Toronto",
) -> pd.DataFrame:
    """
    Full end-to-end pipeline: fetch → classify → parse → dedup → DataFrame.

    Parameters
    ----------
    provider         : Any authenticated EmailProvider.
    fetch_query      : Override the built-in job-hunting search query.
    max_emails       : Maximum number of emails to retrieve.
    after_date       : Ignore emails received before this date.
    min_confidence   : Drop emails whose classification score is below this (0–1).
    deduplicate      : Merge email threads for the same application into one row.
    include_metadata : Add Email ID / Subject / Confidence debug columns.
    local_timezone   : Timezone string for date display, e.g. 'America/Toronto'.

    Returns
    -------
    pd.DataFrame with columns: Job Title, Company, Date Applied, Date Last Updated, Status
    """
    logger.info("=== run_tracker starting ===")

    fetcher    = EmailFetcher(provider, max_results=max_emails,
                              **(({"default_query": fetch_query}) if fetch_query else {}))
    classifier = JobEmailClassifier(min_confidence=min_confidence)
    parser     = JobEmailParser()

    # 1. Fetch
    raw_emails = fetcher.fetch(after_date=after_date)

    # 2. Classify
    classified = classifier.filter(raw_emails)

    # 3. Parse
    applications = [parser.parse(email, conf) for email, conf in classified]

    # 4. Deduplicate
    if deduplicate:
        applications = deduplicate_applications(applications)

    # 5. Build DataFrame
    df = build_dataframe(
        applications,
        include_metadata=include_metadata,
        tz_localize=local_timezone,
    )

    logger.info("Done - %d applications in DataFrame.", len(df))
    return df

print("✅ run_tracker defined. All functions ready - proceed to Section 4.")

✅ run_tracker defined. All functions ready - proceed to Section 4.


---
## 4. Run the Tracker

Authenticate, run the pipeline, and inspect results.
Make sure you've filled in **Section 2. Configuration** before running these cells.

In [14]:
# ── 4.1  Authenticate ─────────────────────────────────────────────────────────
# Runs the browser / device-code OAuth flow.
# After the first run the token is cached, so subsequent runs skip this step.

if EMAIL_PROVIDER == "gmail":
    provider = GmailProvider(
        credentials_path=GMAIL_CREDENTIALS_PATH,
        token_path=GMAIL_TOKEN_PATH,
    )
elif EMAIL_PROVIDER == "outlook":
    provider = OutlookProvider(
        client_id=OUTLOOK_CLIENT_ID,
        tenant_id=OUTLOOK_TENANT_ID,
    )
else:
    raise ValueError(f"Unknown EMAIL_PROVIDER: {EMAIL_PROVIDER!r}")

provider.authenticate()
print(f"✅ Authenticated with {EMAIL_PROVIDER}.")

FileNotFoundError: [Errno 2] No such file or directory: 'credentials.json'

In [ ]:
# ── 4.2  Run the full pipeline ────────────────────────────────────────────────

df = run_tracker(
    provider         = provider,
    max_emails       = MAX_EMAILS,
    after_date       = AFTER_DATE,
    min_confidence   = MIN_CONFIDENCE,
    deduplicate      = DEDUPLICATE,
    include_metadata = INCLUDE_METADATA,
    local_timezone   = LOCAL_TZ,
)

print(f"\n✅ Found {len(df)} job applications.")
df

In [ ]:
# ── 4.3  Status breakdown ─────────────────────────────────────────────────────

print("Applications by status:")
display(df["Status"].value_counts().rename_axis("Status").reset_index(name="Count"))

In [ ]:
# ── 4.4  Filter to a specific status ──────────────────────────────────────────
# Change the value below to any status label from the breakdown above.

FILTER_STATUS = "Interview Scheduled"   # e.g. "Rejected", "Offer Received", "Applied"

filtered = df[df["Status"] == FILTER_STATUS]
print(f"{len(filtered)} applications with status '{FILTER_STATUS}':")
display(filtered)

In [ ]:
# ── 4.5  Save to CSV ──────────────────────────────────────────────────────────

if SAVE_CSV:
    df.to_csv(CSV_PATH, index=False)
    print(f"✅ Saved {len(df)} rows → {CSV_PATH}")
else:
    print("CSV export skipped (SAVE_CSV = False).")